# KG1 V199B H100 baseline-gated continuation

Very conservative continuation from the exact V194 rank-19 / public 0.86 adapter. This notebook evaluates the V194 baseline before training and blocks any candidate that regresses on the same validation split. It does not submit to Kaggle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import hashlib, pathlib, shutil
V194_RANK19_ZIP_SHA256 = '49886191bf9ce92a48106ebfcba407bf9edbe423a4ed8c476d1f6bdfdd210fd8'
V194_RANK19_BOOTSTRAP_TARGET = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V199/baseline_v194_rank19/submission.zip')
V194_RANK19_BOOTSTRAP_SOURCES = [
    V194_RANK19_BOOTSTRAP_TARGET,
    pathlib.Path('/content/submission.zip'),
    pathlib.Path('/content/drive/MyDrive/submission.zip'),
    pathlib.Path('/content/drive/MyDrive/Submit/submission.zip'),
    pathlib.Path('/content/drive/MyDrive/Downloads/submission.zip'),
    pathlib.Path('/content/drive/MyDrive/Download/submission.zip'),
    pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V199/submission.zip'),
]

def sha256_path_bootstrap(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bad_candidates = []
for src in V194_RANK19_BOOTSTRAP_SOURCES:
    print('checking V194 source:', src)
    if not src.exists():
        continue
    got = sha256_path_bootstrap(src)
    print('  sha256:', got)
    if got != V194_RANK19_ZIP_SHA256:
        bad_candidates.append((str(src), got))
        continue
    V194_RANK19_BOOTSTRAP_TARGET.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() != V194_RANK19_BOOTSTRAP_TARGET.resolve():
        shutil.copy2(src, V194_RANK19_BOOTSTRAP_TARGET)
    assert sha256_path_bootstrap(V194_RANK19_BOOTSTRAP_TARGET) == V194_RANK19_ZIP_SHA256
    print('V194 rank-19 zip staged:', V194_RANK19_BOOTSTRAP_TARGET)
    break
else:
    detail = '\n'.join(f'  wrong sha: {p} -> {h}' for p, h in bad_candidates)
    raise RuntimeError(
        'V194 rank-19 submission.zip was not found in Colab/Drive.\n'
        f'Expected SHA256: {V194_RANK19_ZIP_SHA256}\n'
        'Put the validated file at /content/submission.zip, /content/drive/MyDrive/submission.zip, '
        '/content/drive/MyDrive/Submit/submission.zip, '
        'or /content/drive/MyDrive/KG1_NVIDIA_V199/baseline_v194_rank19/submission.zip, then rerun this cell.\n'
        + detail
    )


In [ ]:
import hashlib, importlib.util, json, os, pathlib, shutil, subprocess, sys, urllib.request, zipfile
ROOT = pathlib.Path('/content/kg1_v199')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V199')
V198_DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V198')
TOOLS_ROOT = pathlib.Path('/content/kg1_rank19_tools')
V198_PACK = V198_DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK = V198_PACK if V198_PACK.exists() else DRIVE_ROOT / 'kg1_v198_colab_pack.zip'
PACK_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/31d439bc4a9b33b7b3c772d3526149847103a9b1/runs/v198_micro_distill_colab_pack_20260503/kg1_v198_colab_pack.zip'
PACK_SHA256 = 'e61908c0f75018b0d265c3668600170f6fa99a1a4d559508f489cba9cd6b7c93'
MASTER_PACK_SHA256 = '7e3e41b55bb6f5736c3d5325c7b481f3b52ac918eb13c311e9a343f43f6dedca'
APPROVED_PACK_SHA256 = {PACK_SHA256, MASTER_PACK_SHA256}
AAITDADS_ADAPTER = DRIVE_ROOT / 'component_aaitdads_0p86'
HUIKANG_MODEL_CACHE = DRIVE_ROOT / 'component_huikang_default20_tinker'
LINEAGE_ADAPTER = DRIVE_ROOT / 'component_lineage_51997779_adapter'
RANK19_BUILD = DRIVE_ROOT / 'init_adapter_v194_rank19_build'
INIT_ADAPTER = RANK19_BUILD / 'adapter'
V194_RANK19_ZIP_CANDIDATES = [
    pathlib.Path(os.environ['V194_RANK19_ZIP']) if os.environ.get('V194_RANK19_ZIP') else None,
    DRIVE_ROOT / 'baseline_v194_rank19' / 'submission.zip',
    DRIVE_ROOT / 'best_rank19' / 'submission.zip',
    DRIVE_ROOT / 'v194_rank19_submission.zip',
    DRIVE_ROOT / 'submission_v194_rank19.zip',
    DRIVE_ROOT / 'submission.zip',
]
V194_RANK19_ZIP_CANDIDATES = [p for p in V194_RANK19_ZIP_CANDIDATES if p is not None]
ALLOW_V194_REBUILD_FALLBACK = os.environ.get('ALLOW_V194_REBUILD_FALLBACK') == '1'
AAITDADS_ADAPTER_MODEL_SHA256 = '3d16ba908a5c8808624f1abd8fdc2b29f92723f5c874761161c894d7e5759f21'
AAITDADS_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'
LINEAGE_51997779_ZIP_SHA256 = 'a3b64b154a6690a58f2338ba1c405422eadc6e1c1357f662eecb187463dfdeee'
LINEAGE_HUIKANG_MODEL_HANDLE = 'huikang/nemotron-adapter/transformers/default/20'
LINEAGE_51997779_ADAPTER_MODEL_SHA256 = '559fd024f5ffcaff0caceddeaf25c3801009d6cabf247fc8dfccbfaf2addd916'
LINEAGE_51997779_ADAPTER_CONFIG_SHA256 = 'aaced193a997606aebd7eee1a7cfff5083c301c50a3e940de3611ee559374b61'
V194_RANK19_ADAPTER_MODEL_SHA256 = '01259fef943bc16c31d8f7907be076cc987381a6a1bbe732b1b33c2d9f2ea95f'
V194_RANK19_ADAPTER_CONFIG_SHA256 = 'e5499f128fde60d32d0595d427e4fe84d8abe6dbde1d80886c970e8184e4b743'
V194_RANK19_ZIP_SHA256 = '49886191bf9ce92a48106ebfcba407bf9edbe423a4ed8c476d1f6bdfdd210fd8'
V194_RANK19_DESCRIPTION = 'v194 attention-only micro-merge aaitdads98p5 lineage1p5 keep lmhead experts sha49886191 gate-doublecheck-pass'
V194_RANK19_PUBLIC_SCORE = '0.86'
V194_RANK19_RANK = '19/2613'
BEST_RANKING_BASELINE_RULE = 'always_start_from_best_known_kaggle_ranking_submission'
BEST_RANKING_BASELINE = {
    'ref': '52275052',
    'name': 'V194 rank-19',
    'rank': V194_RANK19_RANK,
    'public_score': V194_RANK19_PUBLIC_SCORE,
    'description': V194_RANK19_DESCRIPTION,
    'adapter_model_sha256': V194_RANK19_ADAPTER_MODEL_SHA256,
    'zip_sha256': V194_RANK19_ZIP_SHA256,
}
assert BEST_RANKING_BASELINE_RULE == 'always_start_from_best_known_kaggle_ranking_submission'
assert BEST_RANKING_BASELINE['rank'] == '19/2613', BEST_RANKING_BASELINE
assert BEST_RANKING_BASELINE['adapter_model_sha256'] == V194_RANK19_ADAPTER_MODEL_SHA256
assert BEST_RANKING_BASELINE['zip_sha256'] == V194_RANK19_ZIP_SHA256
FORBIDDEN_INIT_PATH_FRAGMENTS = ('KG1_NVIDIA_V195/output_v195', 'KG1_NVIDIA_V198/output_v198/final_adapter', 'init_adapter_0p86_aaitdads', 'checkpoint-55', 'checkpoint-75', 'checkpoint-110')
OUT_BASE = DRIVE_ROOT / 'output_v199b_h100_baseline_gated_10'

def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def adapter_ready(path, min_model_bytes=1_000_000):
    cfg = path / 'adapter_config.json'
    model = path / 'adapter_model.safetensors'
    if not cfg.exists() or not model.exists():
        return False
    if cfg.stat().st_size < 100 or model.stat().st_size < min_model_bytes:
        return False
    json.loads(cfg.read_text(encoding='utf-8'))
    return True

def pip_install_quiet(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def purge_modules(*prefixes):
    for name in list(sys.modules):
        if any(name == prefix or name.startswith(prefix + '.') for prefix in prefixes):
            del sys.modules[name]

def install_kagglehub_runtime():
    # kagglehub 1.0.1 imports get_web_endpoint, which only exists in newer kagglesdk.
    pip_install_quiet(['--upgrade', '--force-reinstall', 'kagglesdk==0.1.23', 'kagglehub==1.0.1'])
    purge_modules('kagglehub', 'kagglesdk')
    import importlib.metadata as md
    import kagglesdk.kaggle_env as ke
    assert hasattr(ke, 'get_web_endpoint'), f'Broken kagglesdk {md.version("kagglesdk")}: missing get_web_endpoint'
    import kagglehub
    assert hasattr(kagglehub, 'dataset_download') and hasattr(kagglehub, 'model_download'), 'Broken kagglehub install'
    print('Kaggle runtime:', 'kagglehub', md.version('kagglehub'), 'kagglesdk', md.version('kagglesdk'))
    return kagglehub

def ensure_aaitdads_component():
    AAITDADS_ADAPTER.mkdir(parents=True, exist_ok=True)
    cfg = AAITDADS_ADAPTER / 'adapter_config.json'
    model = AAITDADS_ADAPTER / 'adapter_model.safetensors'
    if adapter_ready(AAITDADS_ADAPTER, min_model_bytes=4_000_000_000):
        cfg_ok = sha256_path(cfg) == AAITDADS_ADAPTER_CONFIG_SHA256
        model_ok = sha256_path(model) == AAITDADS_ADAPTER_MODEL_SHA256
        if cfg_ok and model_ok:
            return AAITDADS_ADAPTER
        print('Existing aaitdads component SHA mismatch; deleting and redownloading.')
        for p in [cfg, model]:
            if p.exists():
                p.unlink()
    kagglehub = install_kagglehub_runtime()
    print('Downloading aaitdads/my-0p86-adapter component...')
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_config.json', output_dir=str(AAITDADS_ADAPTER), force_download=True)
    kagglehub.dataset_download('aaitdads/my-0p86-adapter', path='adapter_model.safetensors', output_dir=str(AAITDADS_ADAPTER), force_download=True)
    assert adapter_ready(AAITDADS_ADAPTER, min_model_bytes=4_000_000_000), f'Missing aaitdads component: {AAITDADS_ADAPTER}'
    assert sha256_path(cfg) == AAITDADS_ADAPTER_CONFIG_SHA256, 'aaitdads adapter_config SHA mismatch'
    assert sha256_path(model) == AAITDADS_ADAPTER_MODEL_SHA256, 'aaitdads adapter_model SHA mismatch'
    return AAITDADS_ADAPTER

def configure_kaggle_credentials():
    kaggle_dir = pathlib.Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    drive_json = pathlib.Path('/content/drive/MyDrive/kaggle.json')
    if not kaggle_json.exists() and drive_json.exists():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_json, kaggle_json)
    if not kaggle_json.exists():
        try:
            from google.colab import userdata
            username = userdata.get('KAGGLE_USERNAME')
            key = userdata.get('KAGGLE_KEY')
        except Exception:
            username = key = None
        assert username and key, 'Kaggle credentials missing: add KAGGLE_USERNAME/KAGGLE_KEY secrets or /content/drive/MyDrive/kaggle.json'
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
    os.chmod(kaggle_json, 0o600)
    os.environ['KAGGLE_CONFIG_DIR'] = str(kaggle_dir)
    return kaggle_json

def locate_adapter_dir(root):
    root = pathlib.Path(root)
    candidates = []
    if (root / 'adapter_model.safetensors').exists() and (root / 'adapter_config.json').exists():
        candidates.append(root)
    candidates.extend(p.parent for p in root.rglob('adapter_model.safetensors') if (p.parent / 'adapter_config.json').exists())
    candidates = sorted(set(candidates), key=lambda p: len(str(p)))
    assert candidates, f'No adapter_model.safetensors + adapter_config.json under {root}'
    return candidates[0]

def patch_tinker_cookbook_merge():
    import torch
    import tinker_cookbook.weights._adapter as A
    FORCED_FUSED_RANK = 32

    def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
        delta = B.float() @ A_mat.float()
        U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
        U = U[:, :rank]
        S = S[:rank]
        Vh = Vh[:rank, :]
        sroot = torch.sqrt(S)
        B_new = U * sroot.unsqueeze(0)
        A_new = sroot.unsqueeze(1) * Vh
        return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()

    def patched_merge_fused_projections(fused_model_key, adapter_layer_prefix, components, model_state_shapes, peft_weights, target_modules, profile):
        fused_out_dim = model_state_shapes[fused_model_key][0]
        fused_target_name = fused_model_key.removesuffix('.weight').rsplit('.', 1)[-1]
        component_order = None
        for target, comps in profile.fused_projection_map:
            if target == fused_target_name:
                component_order = comps
                break
        assert component_order is not None
        comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}
        lora_A_parts = []
        comp_slices = []
        merged_rank = 0
        row_offset = 0
        for comp_name in component_order:
            if comp_name not in comp_by_name:
                raise RuntimeError(f'Missing component {comp_name!r} for fused target {fused_model_key!r}')
            lora_A, lora_B = comp_by_name[comp_name]
            r = lora_A.shape[0]
            out_dim = lora_B.shape[0]
            lora_A_parts.append(lora_A)
            comp_slices.append((row_offset, row_offset + out_dim, r))
            row_offset += out_dim
            merged_rank += r
        merged_lora_A = torch.cat(lora_A_parts, dim=0)
        merged_lora_B = torch.zeros(fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device)
        rank_offset = 0
        for i, (row_start, row_end, r) in enumerate(comp_slices):
            _, lora_B = comp_by_name[component_order[i]]
            merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
            rank_offset += r
        final_rank = merged_rank
        if merged_rank > FORCED_FUSED_RANK:
            merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(merged_lora_B, merged_lora_A, FORCED_FUSED_RANK)
            final_rank = FORCED_FUSED_RANK
        peft_target_key = f'{adapter_layer_prefix}.{fused_target_name}.weight'
        A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
        return final_rank

    A._merge_fused_projections = patched_merge_fused_projections
    print('Patched tinker_cookbook fused projection merge:', A._merge_fused_projections.__name__)

def ensure_lineage_component():
    LINEAGE_ADAPTER.mkdir(parents=True, exist_ok=True)
    if adapter_ready(LINEAGE_ADAPTER, min_model_bytes=3_000_000_000):
        cfg_ok = sha256_path(LINEAGE_ADAPTER / 'adapter_config.json') == LINEAGE_51997779_ADAPTER_CONFIG_SHA256
        model_ok = sha256_path(LINEAGE_ADAPTER / 'adapter_model.safetensors') == LINEAGE_51997779_ADAPTER_MODEL_SHA256
        if cfg_ok and model_ok:
            return LINEAGE_ADAPTER
        print('Existing lineage component SHA mismatch; rebuilding from Huikang default/20.')
        shutil.rmtree(LINEAGE_ADAPTER, ignore_errors=True)
    configure_kaggle_credentials()
    pip_install_quiet(['tinker-cookbook @ git+https://github.com/thinking-machines-lab/tinker-cookbook.git@nightly'])
    kagglehub = install_kagglehub_runtime()
    from tinker_cookbook import weights
    patch_tinker_cookbook_merge()
    HUIKANG_MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    print('Downloading Huikang default/20 source model for canonical 51997779 lineage...')
    downloaded = pathlib.Path(kagglehub.model_download(LINEAGE_HUIKANG_MODEL_HANDLE, output_dir=str(HUIKANG_MODEL_CACHE), force_download=False))
    tinker_adapter = locate_adapter_dir(downloaded)
    print('Huikang source adapter:', tinker_adapter)
    shutil.rmtree(LINEAGE_ADAPTER, ignore_errors=True)
    print('Rebuilding canonical 51997779 lineage adapter via Tinker cookbook...')
    weights.build_lora_adapter(base_model='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16', adapter_path=str(tinker_adapter), output_path=str(LINEAGE_ADAPTER))
    assert adapter_ready(LINEAGE_ADAPTER, min_model_bytes=3_000_000_000), f'Invalid rebuilt lineage adapter: {LINEAGE_ADAPTER}'
    lineage_cfg_sha = sha256_path(LINEAGE_ADAPTER / 'adapter_config.json')
    lineage_model_sha = sha256_path(LINEAGE_ADAPTER / 'adapter_model.safetensors')
    print('Canonical lineage adapter sha:', lineage_model_sha)
    assert lineage_cfg_sha == LINEAGE_51997779_ADAPTER_CONFIG_SHA256, f'Lineage adapter_config SHA mismatch: {lineage_cfg_sha}'
    assert lineage_model_sha == LINEAGE_51997779_ADAPTER_MODEL_SHA256, f'Lineage adapter_model SHA mismatch: {lineage_model_sha}'
    return LINEAGE_ADAPTER

def extract_v194_rank19_submission_zip(candidate):
    candidate = pathlib.Path(candidate)
    candidate_sha = sha256_path(candidate)
    assert candidate_sha == V194_RANK19_ZIP_SHA256, f'V194 rank-19 submission.zip SHA mismatch for {candidate}: {candidate_sha}'
    shutil.rmtree(RANK19_BUILD, ignore_errors=True)
    INIT_ADAPTER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidate) as zf:
        members = {name for name in zf.namelist() if not name.endswith('/')}
        expected_members = {'adapter_model.safetensors', 'adapter_config.json'}
        assert members == expected_members, f'Unexpected V194 rank-19 zip members: {sorted(members)}'
        zf.extract('adapter_model.safetensors', INIT_ADAPTER)
        zf.extract('adapter_config.json', INIT_ADAPTER)
    zip_path = RANK19_BUILD / 'submission.zip'
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if candidate.resolve() != zip_path.resolve():
        shutil.copy2(candidate, zip_path)
    cfg = INIT_ADAPTER / 'adapter_config.json'
    model = INIT_ADAPTER / 'adapter_model.safetensors'
    assert adapter_ready(INIT_ADAPTER, min_model_bytes=4_000_000_000), f'Invalid extracted V194 adapter: {INIT_ADAPTER}'
    assert sha256_path(cfg) == V194_RANK19_ADAPTER_CONFIG_SHA256
    assert sha256_path(model) == V194_RANK19_ADAPTER_MODEL_SHA256
    assert sha256_path(zip_path) == V194_RANK19_ZIP_SHA256
    return INIT_ADAPTER

def try_load_v194_rank19_zip():
    print('Searching for exact V194 rank-19 submission.zip candidates...')
    for candidate in V194_RANK19_ZIP_CANDIDATES:
        print('  candidate:', candidate)
        if not candidate.exists():
            continue
        print('Found candidate V194 rank-19 zip:', candidate)
        return extract_v194_rank19_submission_zip(candidate)
    print('No exact V194 rank-19 submission.zip candidate found.')
    return None

def missing_v194_zip_message():
    paths = '\n'.join(f'  - {p}' for p in V194_RANK19_ZIP_CANDIDATES)
    return (
        'Exact V194 rank-19 submission.zip is required before training.\n'
        f'Expected zip SHA256: {V194_RANK19_ZIP_SHA256}\n'
        'Upload the validated V194 rank-19 file to one of these Drive paths, then rerun this cell:\n'
        f'{paths}\n'
        'Automatic Tinker reconstruction is disabled by default because it is not byte-stable in this runtime. '
        'Set ALLOW_V194_REBUILD_FALLBACK=1 only for forensic rebuilds, not production training.'
    )

def ensure_rank19_v194_adapter():
    cfg = INIT_ADAPTER / 'adapter_config.json'
    model = INIT_ADAPTER / 'adapter_model.safetensors'
    zip_path = RANK19_BUILD / 'submission.zip'
    if adapter_ready(INIT_ADAPTER, min_model_bytes=4_000_000_000) and zip_path.exists():
        if sha256_path(model) == V194_RANK19_ADAPTER_MODEL_SHA256 and sha256_path(cfg) == V194_RANK19_ADAPTER_CONFIG_SHA256 and sha256_path(zip_path) == V194_RANK19_ZIP_SHA256:
            return INIT_ADAPTER
        print('Cached V194 rank-19 adapter mismatch; rebuilding.')
        shutil.rmtree(RANK19_BUILD, ignore_errors=True)
    recovered = try_load_v194_rank19_zip()
    if recovered is not None:
        return recovered
    if not ALLOW_V194_REBUILD_FALLBACK:
        raise RuntimeError(missing_v194_zip_message())
    print('ALLOW_V194_REBUILD_FALLBACK=1; using SHA-gated reconstruction path.')
    primary = ensure_aaitdads_component()
    other = ensure_lineage_component()
    TOOLS_ROOT.mkdir(parents=True, exist_ok=True)
    soup_script = TOOLS_ROOT / 'kg1_update_space_soup_stream.py'
    urllib.request.urlretrieve('https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/kg1_update_space_soup_stream.py', soup_script)
    if importlib.util.find_spec('safetensors') is None:
        pip_install_quiet(['safetensors==0.7.0'])
    print('Rebuilding exact V194 rank-19 adapter: 98.5% aaitdads + 1.5% lineage attention-only.')
    subprocess.run([
        sys.executable, str(soup_script),
        '--primary-adapter', str(primary),
        '--other-adapter', str(other),
        '--output-dir', str(RANK19_BUILD),
        '--config-source', str(primary / 'adapter_config.json'),
        '--primary-weight', '0.985',
        '--other-weight', '0.015',
        '--rank', '32',
        '--copy-safe-primary-non-lora',
        '--include-key-regex', r'\.mixer\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj)\.lora_A\.',
    ], check=True)
    manifest = json.loads((RANK19_BUILD / 'update_space_soup_manifest.json').read_text(encoding='utf-8'))
    assert manifest.get('output_adapter_sha256') == V194_RANK19_ADAPTER_MODEL_SHA256, manifest
    assert manifest.get('output_zip_sha256') == V194_RANK19_ZIP_SHA256, manifest
    assert adapter_ready(INIT_ADAPTER, min_model_bytes=4_000_000_000), f'V194 rank-19 adapter was not built: {INIT_ADAPTER}'
    assert sha256_path(cfg) == V194_RANK19_ADAPTER_CONFIG_SHA256
    assert sha256_path(model) == V194_RANK19_ADAPTER_MODEL_SHA256
    assert sha256_path(zip_path) == V194_RANK19_ZIP_SHA256
    return INIT_ADAPTER

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
INIT_ADAPTER = ensure_rank19_v194_adapter()
init_path_text = str(INIT_ADAPTER)
assert not any(fragment in init_path_text for fragment in FORBIDDEN_INIT_PATH_FRAGMENTS), f'Forbidden init adapter lineage: {INIT_ADAPTER}'
init_sha = sha256_path(INIT_ADAPTER / 'adapter_model.safetensors')
print('Confirmed V194 rank-19 adapter sha:', init_sha)
print('Best-ranking baseline rule:', BEST_RANKING_BASELINE_RULE)
print('Confirmed V194 public score/rank:', V194_RANK19_PUBLIC_SCORE, V194_RANK19_RANK)
assert init_sha == V194_RANK19_ADAPTER_MODEL_SHA256, 'Init adapter must be exact V194 rank-19 adapter.'
assert BEST_RANKING_BASELINE['adapter_model_sha256'] == init_sha, 'Init adapter is not the best-ranking baseline.'

if not PACK.exists():
    print('V198 pack not found in Drive; downloading verified pack...')
    urllib.request.urlretrieve(PACK_URL, PACK)
pack_hash = sha256_path(PACK)
print('Pack SHA256:', pack_hash)
assert pack_hash in APPROVED_PACK_SHA256, f'Pack SHA mismatch: {pack_hash}'
if pack_hash != PACK_SHA256:
    print('Using approved legacy V198 pack; fixed training script will be downloaded before training.')

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    zf.extractall(ROOT)
assert (ROOT / 'data/v198/v198_micro_train.strict.jsonl').exists()
assert (ROOT / 'data/v198/v198_micro_val.strict.jsonl').exists()
assert (ROOT / 'scripts/hf_job_train_v90.py').exists()
print('Pack extracted to', ROOT)


In [ ]:
%cd /content/kg1_v199
import importlib.util, os, subprocess, sys
os.environ.setdefault('MAX_JOBS', '4')
os.environ.setdefault('PIP_ROOT_USER_ACTION', 'ignore')

def pip_install(args):
    print('+ pip install', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def pip_uninstall(package_name):
    print('+ pip uninstall -y', package_name)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name], check=False)

def install_if_missing(module_name, args):
    if importlib.util.find_spec(module_name) is None:
        pip_install(args)
    else:
        print(f'{module_name} already installed')

pip_uninstall('torchao')
pip_install(['--upgrade', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja==1.13.0'])
pip_install(['transformers==5.7.0', 'accelerate==1.13.0', 'peft==0.19.1', 'datasets==4.8.5', 'safetensors==0.7.0', 'huggingface_hub==1.13.0', 'sentencepiece==0.2.1', 'protobuf==7.34.1'])
install_if_missing('causal_conv1d', ['causal-conv1d==1.6.1', '--no-build-isolation'])
install_if_missing('mamba_ssm', ['mamba-ssm==2.3.1', '--no-build-isolation'])
assert importlib.util.find_spec('torchao') is None, 'torchao still installed; restart runtime and rerun cells from top'
import causal_conv1d, mamba_ssm
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


In [ ]:
import pathlib, re, shutil, subprocess
gpu_csv = subprocess.check_output(
    'nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits',
    shell=True,
).decode().strip()
print('GPU:', gpu_csv)
parts = [part.strip() for part in gpu_csv.split(',')]
assert len(parts) >= 3, f'Unexpected nvidia-smi output: {gpu_csv}'
gpu_name = parts[0]
gpu_mem_mib = int(parts[1])
driver_version = parts[2]
assert 'H100' in gpu_name, f'Use Colab Pro H100 High-RAM for this notebook; found {gpu_name}'
assert gpu_mem_mib >= 75000, f'H100 memory too small: {gpu_mem_mib} MiB'
meminfo = pathlib.Path('/proc/meminfo').read_text(encoding='utf-8')
host_mem_kib = int(re.search(r'MemTotal:\s+(\d+)', meminfo).group(1))
host_mem_gib = host_mem_kib / 1024 / 1024
disk = shutil.disk_usage('/content')
disk_free_gib = disk.free / 1024**3
print(f'Host RAM: {host_mem_gib:.1f} GiB')
print(f'/content free: {disk_free_gib:.1f} GiB')
print('Driver:', driver_version)
assert host_mem_gib >= 50, f'High-RAM runtime expected; host RAM is only {host_mem_gib:.1f} GiB'
assert disk_free_gib >= 100, f'Need at least 100 GiB free on /content; found {disk_free_gib:.1f} GiB'


In [ ]:
import os, pathlib, shutil, urllib.request
OUT = OUT_BASE
if OUT.exists():
    import datetime
    suffix = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    OUT = pathlib.Path(str(OUT_BASE) + '_' + suffix)
OUT.mkdir(parents=True, exist_ok=True)
print('V199_OUT =', OUT)
FIXED_TRAIN_SCRIPT_URL = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts/hf_job_train_v90.py'
TRAIN_SCRIPT = pathlib.Path('/content/kg1_v199/scripts/hf_job_train_v90.py')
script_text = TRAIN_SCRIPT.read_text(encoding='utf-8') if TRAIN_SCRIPT.exists() else ''
if 'load_peft_weights_with_direct_fallback' not in script_text or 'BASELINE_EVAL_BEFORE_TRAIN' not in script_text:
    print('Runtime has stale hf_job_train_v90.py; downloading PEFT direct-load fixed script...')
    urllib.request.urlretrieve(FIXED_TRAIN_SCRIPT_URL, TRAIN_SCRIPT)
script_text = TRAIN_SCRIPT.read_text(encoding='utf-8')
assert 'load_peft_weights_with_direct_fallback' in script_text
assert 'PEFT_MANUAL_LOAD_METHOD' in script_text
assert 'BASELINE_EVAL_BEFORE_TRAIN' in script_text
assert 'REQUIRE_FINAL_EVAL_LTE_BASELINE' in script_text
os.environ['UPLOAD_TO_HF'] = '0'
os.environ['MODEL_NAME'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
os.environ['DATA_FILE'] = '/content/kg1_v199/data/v198/v198_micro_train.strict.jsonl'
os.environ['VAL_FILE'] = '/content/kg1_v199/data/v198/v198_micro_val.strict.jsonl'
os.environ['INIT_ADAPTER_DIR'] = str(INIT_ADAPTER)
os.environ['INIT_ADAPTER_LOAD_MODE'] = 'manual'
os.environ['PEFT_MANUAL_LOAD_METHOD'] = 'direct'
os.environ['OUTPUT_DIR'] = str(OUT)
os.environ['V199_OUT'] = str(OUT)
os.environ['RUN_ID'] = 'v199b-h100-baseline-gated-v194-rank19-10s'
os.environ['MAX_LENGTH'] = '2048'
os.environ['BATCH_SIZE'] = '16'
os.environ['MICRO_BATCH_SIZE'] = '1'
os.environ['GRADIENT_CHECKPOINTING'] = '1'
os.environ['MAX_STEPS'] = '10'
os.environ['SAVE_EVERY_STEPS'] = '5'
os.environ['EVAL_EVERY_STEPS'] = '5'
os.environ['EVAL_MAX_EXAMPLES'] = '360'
os.environ['LEARNING_RATE'] = '1e-6'
os.environ['FINAL_LEARNING_RATE'] = '3e-7'
os.environ['ABORT_EVAL_LOSS_GT'] = '0'
os.environ['BASELINE_EVAL_BEFORE_TRAIN'] = '1'
os.environ['ABORT_EVAL_RELATIVE_TO_BASELINE_DELTA'] = '0.02'
os.environ['REQUIRE_FINAL_EVAL_LTE_BASELINE'] = '1'
os.environ['MAX_FINAL_EVAL_REGRESSION'] = '0.0'
os.environ['EXPECTED_TRAIN_SHA256'] = '6d2742616300818eb50c54d36019551b24f5b71c607a2b28feda7461a709def0'
os.environ['EXPECTED_VAL_SHA256'] = 'e59c907c6545e5e587097a64762e3e874508e8cd74d85d5c7c79354ebe56e73c'
os.environ['MIN_TRAIN_EXAMPLES'] = '1875'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES'] = '1600'
os.environ['MIN_VAL_EXAMPLES'] = '720'
os.environ['MIN_TOKENIZED_VAL_EXAMPLES'] = '700'
os.environ['TRAINABLE_LORA_MODULES'] = 'in_proj,out_proj,q_proj,k_proj,v_proj,o_proj'
os.environ['MAX_TRAINABLE_PARAM_RATIO'] = '0.035'
!python scripts/hf_job_train_v90.py


Convert and gate the V199 adapters. This still does not submit to Kaggle.


In [ ]:
import json, os, pathlib, subprocess, sys, urllib.request
BASE = 'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/claude/competent-shamir/scripts'
for name in ['kg1_v198_posttrain_gate.py', 'kg1_v199_posttrain_gate.py', 'nemotron_submission_preflight.py', 'kg1_submission_gate.py', 'kg1_v198_final_submit_doublecheck.py']:
    dst = pathlib.Path('/content/kg1_v199/scripts') / name
    print('downloading', name)
    urllib.request.urlretrieve(f'{BASE}/{name}', dst)

!python scripts/kg1_v199_posttrain_gate.py --root /content/kg1_v199 --output-root "$V199_OUT" --fail-on-block

report_path = pathlib.Path(os.environ['V199_OUT']) / 'posttrain_kaggle_gate/v199_posttrain_gate_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
assert report['decision']['ready'], report['decision']
primary_zip = report['decision']['primary_zip']
primary_label = report['decision']['primary_label']
print('primary_label =', primary_label)
print('primary_zip =', primary_zip)
assert primary_label == 'final', f'Blocked: only final adapter can be promoted for V199, got {primary_label}'

preflight_json = pathlib.Path(os.environ['V199_OUT']) / f'{primary_label}_preflight.json'
subprocess.run([sys.executable, 'scripts/nemotron_submission_preflight.py', '--adapter-zip', primary_zip, '--output-json', str(preflight_json), '--fail-on-block'], check=True)

doublecheck_json = pathlib.Path(os.environ['V199_OUT']) / f'{primary_label}_submit_doublecheck.json'
subprocess.run([
    sys.executable, 'scripts/kg1_v198_final_submit_doublecheck.py',
    '--candidate-zip', primary_zip,
    '--expected-label', 'final',
    '--posttrain-report', str(report_path),
    '--preflight-report', str(preflight_json),
    '--output-json', str(doublecheck_json),
    '--fail-on-block',
], check=True)
print('V199 gated candidate ready. No Kaggle submit was performed.')
print('doublecheck:', doublecheck_json)
